In [1]:
import pandas as pd
import re #regular expression
from collections import defaultdict
label_column="class"
expected_features_per_task=18
copy_idx     = {6,7,8,9,10,11,12,13,15,16,17,19,22,25}
graphic_idx  = {2,3,4,5,21,24}
memory_idx   = {1,14,18,20,23}
df=pd.read_csv("data.csv")
all_cols=list(df.columns)
if label_column not in all_cols:
  raise ValueError(f"{label_column} not in columns")
def cols_for_task(task_num,columns):#returns list of columns that end with task number using regex expression
  pattern=re.compile(rf'(?<!\d){re.escape(str(task_num))}$')
  matches=[c for c in columns if pattern.search(c)]
  return matches
# build dataset for a set of task numbers
def build_task_dataset(task_set,df,expected_per_task=expected_features_per_task):
  task_set=sorted(task_set)
  selected_cols=[]
  per_task_report=defaultdict(list)
  for t in task_set:
    cols=cols_for_task(t,df.columns)
    cols_sorted=sorted(cols)
    per_task_report[t]=cols_sorted
    selected_cols.extend(cols_sorted)
#remove duplicates while preserving the order
  seen=set()
  selected_cols_unique=[]
  for c in selected_cols:
    if c not in seen:
      seen.add(c)
      selected_cols_unique.append(c)
  #prepare final dataframe and put label column at the end
  if not selected_cols_unique:
    raise ValueError("No columns found")
  final_cols=selected_cols_unique+[label_column]
  missing=[c for c in final_cols if c not in df.columns]
  if missing:
    raise RuntimeError(f"Missing columns: {missing}")
  return df[final_cols].copy(),per_task_report
#build and save the datasets
df_copy_tasks,report_copy=build_task_dataset(copy_idx,df)
df_graphic_tasks,report_graphic=build_task_dataset(graphic_idx,df)
df_memory_tasks,report_memory=build_task_dataset(memory_idx,df)
#save to csv
df_copy_tasks.to_csv("copy_tasks.csv",index=False)
df_graphic_tasks.to_csv("graphic_tasks.csv",index=False)
df_memory_tasks.to_csv("memory_tasks.csv",index=False)
#X,y ready for SVC
X_copy=df_copy_tasks.drop(columns=[label_column]).values
y_copy=df_copy_tasks[label_column].values
X_graphic=df_graphic_tasks.drop(columns=[label_column]).values
y_graphic=df_graphic_tasks[label_column].values
X_memory=df_memory_tasks.drop(columns=[label_column]).values
y_memory=df_memory_tasks[label_column].values




In [2]:
X=X_graphic
y=y_graphic

In [3]:
from sklearn.preprocessing import StandardScaler #scaling
from sklearn.decomposition import PCA #reduce the dimension
from sklearn.pipeline import Pipeline #chain steps
preprocess=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler())])
X_processing=preprocess.fit_transform(X) #fit and transform

In [4]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

def make_12q_pqc(bandwidth):
    """
    12-qubit feature map using 24 unique parameters (reused for both encoding layers).
    Structure:
      - encoding layer 1: RX(x[0..11]) then RY(x[12..23])
      - entangling layer 1: CZ ring
      - encoding layer 2: reuse RX(x[0..11]) then RY(x[12..23])
      - entangling layer 2: CX ring
    """
    x = ParameterVector("x", 24)# 24 unique parameters
    theta = bandwidth * x # bandwidth scales the parameters
    qc = QuantumCircuit(12)
    #encoding layer 1 (RX then RY)
    for q in range(12):
        qc.rx(theta[q], q)
    for q in range(12):
        qc.ry(theta[12 + q], q)
    #entangling layer 1: CZ ring
    for q in range(11):
        qc.cz(q, q + 1)
    qc.cz(11, 0)
    #encoding layer 2 (reuse same 24 parameters)
    # reset to same parameters (so no index-out-of-range)
    for q in range(12):
        qc.rx(theta[q], q)
    for q in range(12):
        qc.ry(theta[12 + q], q)
    #entangling layer 2: CX ring
    for q in range(11):
        qc.cx(q, q + 1)
    qc.cx(11, 0)
    return qc


In [5]:
from qiskit.primitives import StatevectorSampler
from qiskit import QuantumCircuit
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.model_selection import ShuffleSplit,train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
pqc=QuantumCircuit()
#let fidelity kernel use the internal default statevector simulator (noiseless simulator)
def make_12q_kernel(bandwidth): #FidelityQuantumKernel using 6q pqc and aer based estimator
    pqc=make_12q_pqc(bandwidth)
    qkernel=FidelityQuantumKernel(feature_map=pqc)
    return qkernel


In [6]:
import numpy as np
from sklearn.model_selection import ShuffleSplit, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
outer_cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
bandwidth_grid = np.linspace(0.1, 2.0, 10) # Reduced grid size to 10 for speed
test_accs_12q = []
for i, (trainval_idx, test_idx) in enumerate(outer_cv.split(X_processing, y)):
    X_trainval, X_test = X_processing[trainval_idx], X_processing[test_idx]
    y_trainval, y_test = y[trainval_idx], y[test_idx]
    train_idx, val_idx = train_test_split(
        np.arange(len(X_trainval)),
        test_size=0.25,
        stratify=y_trainval,
        random_state=42
    )
    best_bw = None
    best_val_acc = -np.inf
    for bw in bandwidth_grid:
        qkernel = make_12q_kernel(bw)
        K_train = qkernel.evaluate(X_trainval[train_idx], X_trainval[train_idx])
        K_val = qkernel.evaluate(X_trainval[val_idx], X_trainval[train_idx])
        clf = SVC(kernel="precomputed", C=10.0, random_state=42)
        clf.fit(K_train, y_trainval[train_idx])
        val_acc = accuracy_score(y_trainval[val_idx], clf.predict(K_val))
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_bw = bw
    qkernel_best = make_12q_kernel(best_bw)
    K_train_full = qkernel_best.evaluate(X_trainval, X_trainval)
    K_test = qkernel_best.evaluate(X_test, X_trainval)
    clf_final = SVC(kernel="precomputed", C=10.0, random_state=42)
    clf_final.fit(K_train_full, y_trainval)
    test_acc = accuracy_score(y_test, clf_final.predict(K_test))
    test_accs_12q.append(test_acc)
    print(f"Split {i+1}: bw={best_bw:.2f}, acc={test_acc*100:.2f}%")
mean_acc_12q = np.mean(test_accs_12q) * 100
std_acc_12q = np.std(test_accs_12q) * 100
print(f"\n12q Mean Accuracy: {mean_acc_12q:.2f}% ± {std_acc_12q:.2f}%")

Split 1: bw=0.73, acc=74.29%
Split 2: bw=0.10, acc=85.71%
Split 3: bw=0.10, acc=77.14%
Split 4: bw=0.10, acc=85.71%
Split 5: bw=0.31, acc=80.00%

12q Mean Accuracy: 80.57% ± 4.57%
